# CodeGen — Group 45
## Step 3: Fine-tuning — teach the model Rust with LoRA, then re-measure

**What we do here:** take our 194 validated pairs (`pairs.jsonl`), **fine-tune** codegen-350M-multi
on them using **LoRA** (a cheap adapter that fits a free GPU), then re-run the Step-1 harness on
**HumanEval-Rust** and compare against our **~1.3% baseline**.

**The question this answers:** did teaching the model on our validated Rust solutions make it
better at Rust? Even a small jump above 1.3% proves the whole loop (data → train → re-evaluate).

**Leakage check:** we train on MBPP-derived pairs and evaluate on HumanEval-Rust — disjoint sets.

**To run:** needs a **GPU** (`Runtime → Change runtime type → T4 GPU`). Upload `pairs.jsonl`
(Files panel on the left) or mount Drive, then `Runtime → Run all`.


## 1. Install Rust + libraries
Same rule as before — we do **not** upgrade `torch` (it breaks Colab's torchvision). We add
`peft` for LoRA, and remove Colab's old `torchao` (newer peft rejects it; we don't use it).
**If you see a torchao error, just `Runtime → Restart session` and Run all again.**

In [1]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version
# IMPORTANT: do NOT add `torch` here.
!pip install -q -U datasets transformers accelerate peft
# Colab ships an old torchao (0.10) that newer peft rejects; we don't use it, so remove it.
!pip uninstall -q -y torchao
print("setup done")


warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.96.0 (ac68faa20 2026-05-25)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.96.0 (ac68fa

## 2. Load our validated pairs
Upload `pairs.jsonl` via the Files panel (left sidebar), or mount Drive and copy it in.

In [3]:
import os
# Option: mount Drive instead of uploading
from google.colab import drive; drive.mount("/content/drive")
import shutil; shutil.copy("/content/drive/MyDrive/CodeGen_Group45/pairs_v2.jsonl", "pairs_v2.jsonl")

assert os.path.exists("pairs_v2.jsonl"), "Upload pairs.jsonl first (Files panel) or mount Drive."

from datasets import load_dataset
data = load_dataset("json", data_files="pairs_v2.jsonl", split="train")
print(len(data), "validated pairs loaded")
print("\n=== one Rust solution we will train on ===\n", data[0]["rust_solution"][:300])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Generating train split: 0 examples [00:00, ? examples/s]

1202 validated pairs loaded

=== one Rust solution we will train on ===
 /// Write a rsthon function to identify non-prime numbers.
fn is_not_prime(n: isize) -> bool {
    let limit = (n as f64).sqrt() as isize + 1;
    for i in 2..limit {
        if n % i == 0 {
            return true;
        }
    }
    false
}


## 3. Load the base model and attach LoRA
We load codegen-350M-multi and wrap it with a small **LoRA adapter**. Only the adapter trains —
the 350M base stays frozen — so it fits easily on a T4. `print_trainable_parameters()` shows how
tiny the trained part is (usually <1%).

In [4]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from google.colab import drive

# Mount Drive (safe to run even if already mounted)
drive.mount("/content/drive")

# EDIT this to the EXACT folder we saved in Step 2.5:
BASE = "/content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-base"

assert os.path.isdir(BASE), f"Not found: {BASE} — fix the path to match your Drive"
print("Using Rust-aware base:", BASE)
print("Folder contents:", os.listdir(BASE))

# tokenizer: from the saved folder if it's there, else fall back to the original
try:
    tok = AutoTokenizer.from_pretrained(BASE)
except Exception:
    tok = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-multi")
tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE)
model.config.pad_token_id = tok.eos_token_id

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules=["qkv_proj", "out_proj"])
model = get_peft_model(model, lora)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.print_trainable_parameters()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Rust-aware base: /content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-base
Folder contents: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

trainable params: 1,966,080 || all params: 358,678,528 || trainable%: 0.5481


## 4. Tokenize and train (with prompt-masking)
We train the model to produce the Rust **body** given the prompt. Important detail: we **mask the
prompt** from the loss (`labels = -100` on the prompt tokens) so the model learns to *write the
body*, not to reproduce the prompt. We also use a gentle learning rate so generation doesn't
collapse. 194 examples × 3 epochs trains in a few minutes on a T4.

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

def build_example(ex):
    full = ex["rust_solution"] + tok.eos_token
    full_ids   = tok(full, truncation=True, max_length=512)["input_ids"]
    prompt_ids = tok(ex["rust_prompt"], truncation=True, max_length=512)["input_ids"]
    k = min(len(prompt_ids), len(full_ids))
    labels = [-100]*k + full_ids[k:]              # train ONLY on the body, ignore the prompt
    return {"input_ids": full_ids, "attention_mask": [1]*len(full_ids), "labels": labels}

tok_ds   = data.map(build_example, remove_columns=data.column_names)
collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100, padding=True)

args = TrainingArguments(
    output_dir="ckpt",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=1e-4,                # gentle LR so generation doesn't collapse
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
trainer.train()
print("\ntraining done")


Map:   0%|          | 0/1202 [00:00<?, ? examples/s]

Step,Training Loss
10,0.915040
20,0.740404
30,0.831283
40,0.692733
50,0.717717
60,0.734111
70,0.648715
80,0.664470
90,0.640134
100,0.622251



training done


## 5. Save the LoRA adapter
This is the small file we keep (and commit to the repo). It can be re-attached to the base model anytime.

In [6]:
model.save_pretrained("codegen350m-rust-lora")
print("adapter saved to ./codegen350m-rust-lora")
# Optional: copy to Drive
import shutil; shutil.copytree("codegen350m-rust-lora", "/content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-lora", dirs_exist_ok=True)


adapter saved to ./codegen350m-rust-lora


'/content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-lora'

## 6. Re-measure on HumanEval-Rust (the moment of truth)
We rebuild the Step-1 harness and run the **fine-tuned** model on the same 156 HumanEval-Rust
problems, then compare to the ~1.3% baseline. (Takes a few minutes — compiles + runs every output.)

In [7]:
import subprocess, tempfile, os
from collections import Counter

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src, binp = os.path.join(wd, "main.rs"), os.path.join(wd, "prog")
        open(src, "w").write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp], capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"
        return "pass" if r.returncode == 0 else "run_fail"

def load_rs(cfg):
    try:    return load_dataset("nuprl/MultiPL-E", cfg, split="test")
    except Exception: return load_dataset("nuprl/MultiPL-E", cfg, split="test", trust_remote_code=True)

eval_ds = load_rs("humaneval-rs")   # disjoint from our MBPP training data

def trim_to_body(text):
    # The prompt already opened the function brace (depth 1). Cut right before the
    # brace that closes the function, so the test block supplies the only closing brace.
    depth = 1
    for i, ch in enumerate(text):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[:i]
    return text

model.eval()
def model_completion(ex, max_new_tokens=256):
    inputs = tok(ex["prompt"], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

statuses = []
for ex in eval_ds:
    statuses.append(evaluate_one(ex["prompt"], model_completion(ex), ex["tests"]))
acc = 100 * sum(s == "pass" for s in statuses) / len(statuses)

print("="*48)
print(f"Vanilla baseline      : ~1.3 %")
print(f"Fine-tuned (LoRA)     : {acc:.1f} %")
print(f"Breakdown             : {dict(Counter(statuses))}")
print("="*48)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

humaneval-rs/test-00000-of-00001.parquet:   0%|          | 0.00/75.3k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/156 [00:00<?, ? examples/s]

Vanilla baseline      : ~1.3 %
Fine-tuned (LoRA)     : 1.9 %
Breakdown             : {'compile_error': 118, 'run_fail': 35, 'pass': 3}


In [8]:
def model_completion(ex, max_new_tokens=512):     # was 256
    inputs = tok(ex["prompt"], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return trim_to_body(tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

import subprocess, tempfile, os
from collections import Counter
statuses, errs = [], []
for ex in eval_ds:
    program = ex["prompt"] + model_completion(ex) + ex["tests"]
    with tempfile.TemporaryDirectory() as wd:
        src = os.path.join(wd, "m.rs"); open(src, "w").write(program)
        c = subprocess.run(["rustc", src, "-o", os.path.join(wd,"p")], capture_output=True, text=True)
        if c.returncode != 0:
            statuses.append("compile_error"); errs.append(c.stderr)
        else:
            r = subprocess.run([os.path.join(wd,"p")], capture_output=True, text=True, timeout=10)
            statuses.append("pass" if r.returncode==0 else "run_fail")
print(dict(Counter(statuses)), "acc:", round(100*statuses.count("pass")/len(statuses),1), "%")
for e in errs[:5]:
    first = next((l for l in e.splitlines() if "error" in l), "")
    print("  ", first[:150])

{'compile_error': 117, 'run_fail': 36, 'pass': 3} acc: 1.9 %
   error[E0599]: no method named `some` found for struct `Map<I, F>` in the current scope
   error[E0765]: unterminated double quote string
   error: expected one of `:`, `;`, `=`, `@`, or `|`, found `}`
   error[E0599]: no method named `mean` found for type `f64` in the current scope
   error[E0425]: cannot find value `y` in this scope


In [ ]:
ex = eval_ds[3]
inputs = tok(ex["prompt"], return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
raw = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("RAW (untrimmed):", repr(raw[:300]))
print("\nTRIMMED completion:", repr(model_completion(ex)))

RAW (untrimmed): '    let mut result = false;\n    for i in operations {\n        if operations[i].balance < 0 {\n            result = true;\n            break;\n        }\n    }\n    result\n}'

TRIMMED completion: '    let mut result = false;\n    for i in operations {\n        if operations[i].balance < 0 {\n            result = true;\n            break;\n        }\n    }\n    result\n'


## What we achieved (and what's next)
- We **fine-tuned** codegen-350M-multi on our validated Rust pairs and **re-measured** on the
  same benchmark — a clean, apples-to-apples comparison to the 1.3% baseline.
- We now have: vanilla **vs.** fine-tuned. Put both numbers in the results table.

**If the number went up — even a little — the loop works.** That is the core deliverable.

**Next moves to push it further:**
1. **More data:** add Rosetta Code pairs and generate more, then retrain.
2. **Python→Rust variant:** feed the Python as input at eval time to measure true translation
   (not just generation).
3. **RAG:** retrieve similar examples at answer-time on top of the fine-tuned model.
4. **Error analysis:** save a few failures and note *why* (great for the report).

**Summary:** "We built a validated dataset, fine-tuned with LoRA, and measured a
change from the 1.3% baseline on a held-out benchmark — the full data→train→evaluate loop runs."
